This code combines uses an Excel-based analysis/workstream tracker I created with the below code to automatically craft standardized exhibits based on the output of each selected analysis.

### Code Setup - Import Needed Libraries and Set Local Variables if Needed

In [ ]:
#Python 3.13.5
import base64 #Native
import os #Native
import pandas as pd #2.3.0
from pathlib import Path #Native
from PyPDF2 import PdfMerger #3.0.1
import re #2.2.1
from selenium import webdriver #4.38.0
from selenium.webdriver.chrome.service import Service #4.38.0
from selenium.webdriver.chrome.options import Options #4.38.0
import win32com.client #Native
from PIL import ImageGrab #11.3.0
import shutil 
import warnings

#insert path to overall output folder
output_path = r"INSERT PATH TO EXHIBITS" 
#insert path to tracker excel file
Tracker_path = r"INSERT PATH TO TRACKER EXCEL FILE" 

____________

In [10]:
warnings.simplefilter(action='ignore', category=UserWarning)
Exhibit_Data = pd.read_excel(fr"{Tracker_path}", sheet_name="Workstream Tracker", dtype={'LABEL_NUMBER': str, 'Unique_ID': str})
Exhibit_Data = Exhibit_Data.sort_values(by=['ORDER_NUMBER'])
Source_Information = pd.read_excel(fr"{Tracker_path}", sheet_name="Source Tracker")
options = webdriver.ChromeOptions()
options.add_argument("--headless=new")  
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--hide-scrollbars")
options.add_argument("--window-size=1080,1920")
options.add_argument("--force-device-scale-factor=2")

try:
    if os.path.exists(fr"{output_path}"):
        shutil.rmtree(fr"{output_path}")
except Exception as e:
    print(f"\tError creating output directory for deletion: {e}. Please check if the 'path' Link is valid.")
    pass

html_content = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <style>
        @media print{
            table {page-break-inside: avoid;}
            img {page-break-inside: avoid;
                break-inside: avoid;}
            .page-break {page-break-before: always;
                break-before: page}
        }
        body {
            font-family: 'Times New Roman', serif;
            font-size: 12px;
        }
        table {
            width: 100%; 
            margin-left: 5px;
            border-collapse: collapse;
        }
        th, td {
            padding:2px,5px;
            vertical-align: top; 
        }
        h4 {
            text-align: center; 
            font-weight:normal;
            font-size: 16px;
            font-family: 'Times New Roman', serif;
        }
        notetext {
            font-weight: bold;
            text-decoration: underline;
            margin-bottom:5px;
            margin-left:22px;
        }
    </style>
</head>
<body> {{TITLE}} {{EXHIBIT}} {{NOTES}} {{SOURCES}} </body>
</html>"""
 
class OrderedSet(list):
    def add(self, item):
        if item not in self:
            super().append(item)
Exhibits_PDFs = OrderedSet()

def create_exhibit(row_data):
    # SKIP IF NO LABEL
    if len(str(row_data.ORDER_NUMBER)) == 0 or pd.isna(row_data.ORDER_NUMBER):
        return
    
    # FORMAT LABEL NUMBER
    def format_number(n):
        return str(int(n)) if n == int(n) else str(n)
    label_number = format_number(str(row_data.LABEL_NUMBER))
    id_number = format_number(str(row_data.Unique_ID))

    #DELETE PREVIOUS OUTPUT FOLDER
    try:
        exhibit_image_folder = os.path.dirname(row_data.ANALYSIS_OUTPUT_LINK)
        if os.path.exists(fr"{exhibit_image_folder}\Exhibit Output"):
            shutil.rmtree(fr"{exhibit_image_folder}\Exhibit Output")
    except Exception as e:
        print(f"\tError finding exhibit image folder for deletion: {e}. Please check if the Exhibit Image Link is valid.")
        pass
    
    # CREATE OUTPUT FOLDERS
    try:
        exhibit_image_folder = os.path.dirname(row_data.ANALYSIS_OUTPUT_LINK)
        Path(fr"{exhibit_image_folder}\Exhibit Output").mkdir(parents=True, exist_ok=True)
    except Exception as e:
        print(f"\tError finding exhibit image folder: {e}. Please check if the Exhibit Image Link is valid.")
        pass
    try:
        os.makedirs(fr"{output_path}", exist_ok=True)
        os.makedirs(fr"{output_path}\{str(row_data.LABEL)} {str(label_number)}", exist_ok=True)
    except Exception as e:
        print(f"\tError creating output directory: {e}. Please check if the 'path' Link is valid.")
        pass

    # LOG CREATION
    print(f"Creating {str(row_data.LABEL)} {label_number}")
    row_html_content = html_content

    # EXCEL TO PDF TO IMAGE
    def is_excel_file(file_path):
        excel_extensions = ['.xls', '.xlsx', '.xlsm', '.xlsb', '.xltx', '.xltm']
        return any(file_path.lower().endswith(ext) for ext in excel_extensions)
    
    if is_excel_file(str(row_data.ANALYSIS_OUTPUT_LINK)):
        try:
            excel = win32com.client.Dispatch("Excel.Application")
            excel.Visible = False # Set to False for background operation
            workbook = excel.Workbooks.Open(str(row_data.ANALYSIS_OUTPUT_LINK))
            sheet = workbook.Sheets(str(row_data.ANALYSIS_OUTPUT_LINK_SHEET))  
            excel.ActiveWindow.View = 1
            title_rows = sheet.PageSetup.PrintTitleRows
            second_number = int(title_rows.split(":")[1].replace("$", ""))
            used = sheet.UsedRange
            first_row = used.Row
            last_row = used.Row + used.Rows.Count - 1
            first_col = used.Column
            last_col = used.Column + used.Columns.Count - 1
            page_breaks = sheet.HPageBreaks
            break_rows = [first_row]  
            for pb in page_breaks:
                break_rows.append(pb.Location.Row)
            break_rows.append(last_row + 1)  # End marker
            page_number = 1
            ranges = []
            for i in range(len(break_rows) - 1):
                start = 1
                end = break_rows[i + 1] - 1  
                page_range = sheet.Range(
                    sheet.Cells(start, first_col),
                    sheet.Cells(end, last_col)
                )
                ranges.append(page_range)
                page_number += 1
            used.Select()
            retries, success = 100, False
            while not success:
                try:
                    used.CopyPicture(1, 2)
                    im = ImageGrab.grabclipboard()
                    im.save(fr"{exhibit_image_folder}\Exhibit Output\{id_number} Full Table.png")
                    success = True
                except:
                    retries -= 1
                    if retries == 0: continue
            previously_used_row = 0
            for i, r in enumerate(ranges, start=1):
                if i > 1:  
                    first_row = r.Rows 
                    sheet.Rows(f"{second_number+1}:{previously_used_row}").EntireRow.Hidden = True
                previously_used_row = r.Rows.Count
                r.Select()
                retries, success = 100, False
                while not success:
                    try:
                        r.CopyPicture(1, 2)
                        im = ImageGrab.grabclipboard()
                        im.save(fr"{exhibit_image_folder}\Exhibit Output\{id_number} Inter_{str(i)}.png")
                        success = True
                    except:
                        retries -= 1
                        if retries == 0: continue
            workbook.Close(SaveChanges=False)
            excel.Quit()
        except Exception as e:
            print(f"\tError converting Excel to images: {e}. Please check if the 'Exhibit Output Link' and 'Exhibit Output Link Sheet' are valid.")
            pass

    # TITLE
    if len(str(row_data.TITLE)) > 0 and not pd.isna(row_data.TITLE):
        Title_Info = re.findall(r'\{(.*?)\}', str(row_data.TITLE))
        replacement_string = ""
        for x in range(len(Title_Info)):
            if x != range(len(Title_Info)):
                replacement_string += Title_Info[x] + '<br>'
            else:
                replacement_string += Title_Info[x] 
        row_html_content = row_html_content.replace("{{TITLE}}",'<div style="text-align:center;"><h4>' + f'<a style=font-weight:bold>{str(row_data.LABEL)} {str(label_number)}</a>: ' + replacement_string + '</h4></div>')
    else:
        row_html_content = row_html_content.replace("{{TITLE}}",'<div style="text-align:center;"><h4>' + f"{str(row_data.LABEL)} {str(label_number)}</h4></div>")

    # EXHIBIT IMAGE
    if is_excel_file(str(row_data.ANALYSIS_OUTPUT_LINK)):
        image_files = []
        for file in os.listdir(exhibit_image_folder + fr"\Exhibit Output"):
            if file.startswith(f"{id_number} Inter_") and file.endswith(".png"):
                image_files.append(file)
        image_files.sort()
        exhibit_images_html = ""
        for image_file in image_files:
            exhibit_images_html += fr'<img src="file:\\\{exhibit_image_folder}\Exhibit Output\{image_file}" alt="Exhibit" style="width:100%;height:auto;display: block; margin-left: auto;margin-right: auto;"><br>'
        row_html_content = row_html_content.replace("{{EXHIBIT}}", exhibit_images_html)
    else:
        if len(str(row_data.ANALYSIS_OUTPUT_LINK)) > 0 and not pd.isna(row_data.ANALYSIS_OUTPUT_LINK):
            row_html_content = row_html_content.replace("{{EXHIBIT}}", fr'<img src="file:\\\{row_data.ANALYSIS_OUTPUT_LINK}" alt="Exhibit" style="width:100%;height:auto;display: block; margin-left: auto;margin-right: auto;"><br>')
        else:
            row_html_content = row_html_content.replace("{{EXHIBIT}}", "")

    # NOTES
    if len(str(row_data.NOTES)) > 0 and not pd.isna(row_data.NOTES):
        Notes_Info = re.findall(r'\{(.*?)\}', str(row_data.NOTES))
        replacement_string = ""
        for x in range(len(Notes_Info)):
            replacement_string += '<tr><td>[' +str(x+1) + ']</td><td>' + Notes_Info[x] + '</td></tr>'
        row_html_content = row_html_content.replace("{{NOTES}}",'<br><notetext>Notes:</notetext><table style="display:flex"><table style="display:flex"><colgroup><col style="width: 5%;text-align:right"><col style="width: 95%;text-align:left"></colgroup><tbody>' + replacement_string + '</tbody></table>')
    else:
        row_html_content = row_html_content.replace("{{NOTES}}","")

    # PREPARE SOURCES
    Source_Dict = {}
    for row in Source_Information.iterrows():
        Source_Dict[row[1]['Unique ID']] = row[1]['Official Citation']
    for x in Source_Dict:
        if pd.isna(Source_Dict[x]):
            Source_Dict[x] = []
        else:
            Source_Dict[x] = [item.strip() for item in Source_Dict[x].split(",")]

    Predecessor_Dict = {}
    for row in Source_Information.iterrows():
        Predecessor_Dict[row[1]['Unique ID']] = row[1]['Predecessors']
    for x in Predecessor_Dict:
        if pd.isna(Predecessor_Dict[x]):
            Predecessor_Dict[x] = []
        else:
            Predecessor_Info = re.findall(r'\{(.*?)\}', str(Predecessor_Dict[x]))
            Predecessor_Dict[x] = [item.strip() for item in Predecessor_Info]

    for x in Predecessor_Dict:
        for y in Predecessor_Dict[x]:
            Source_Dict[x].extend(Source_Dict[y])

    for x in Predecessor_Dict:
        for y in Predecessor_Dict[x]:
            Source_Dict[x].extend(Source_Dict[y])

    for x in Source_Dict:
        Source_Dict[x] = list(set(Source_Dict[x]))
        Source_Dict[x].sort()

    # SOURCES
    if len(str(row_data.Sources)) > 0 and not pd.isna(row_data.Sources):
        replacement_string = ""
        Sources_Info = re.findall(r'\{(.*?)\}', str(row_data.Sources))
        replacement_string = ""
        Exhibit_Sources = []
        for i, x in enumerate(Sources_Info):
            Exhibit_Sources.extend(Source_Dict[x])
        Exhibit_Sources = list(set(Exhibit_Sources))
        Exhibit_Sources.sort()
        for x in range(len(Exhibit_Sources)):
            replacement_string += '<tr><td>[' +chr(65+x) + ']</td><td>' + Exhibit_Sources[x] + '.</td></tr>'
        row_html_content = row_html_content.replace("{{SOURCES}}",'<br><notetext>Sources:</notetext><table style="display:flex"><colgroup><col style="width: 5%;text-align:right"><col style="width: 95%;text-align:left"></colgroup><tbody>' + replacement_string + '</tbody></table>')
    else:
        row_html_content = row_html_content.replace("{{SOURCES}}","")
    
    # WRITE HTML FILE
    try:
        with open(fr"{output_path}\{str(row_data.LABEL)} {str(label_number)}\{str(row_data.LABEL)} {str(label_number)}.html", "w") as file:
            file.write(row_html_content)
    except Exception as e:
        print(f"\tError writing exhibit output HTML file to overall output folder: {e}. Please check if the 'output path' Link is valid.")
        pass
    try:
        with open(fr"{exhibit_image_folder}\Exhibit Output\{id_number}.html", "w") as file:
            file.write(row_html_content)
    except Exception as e:
        print(f"\tError writing exhibit output HTML file to exhibit image folder: {e}. Please check if the Exhibit Image Link is valid.")
        pass

    # CONVERT HTML TO IMAGES AND PDF
    driver = webdriver.Chrome(options=options)
    driver.get(fr"{output_path}\{str(row_data.LABEL)} {str(label_number)}\{str(row_data.LABEL)} {str(label_number)}.html")

    # CREATE IN-REPORT IMAGE (sans Header and Footer)
    try:
        scroll_height = driver.execute_script("return document.body.scrollHeight")
        scroll_height += 190 
        driver.set_window_size(1080, scroll_height)
        screenshot = driver.execute_cdp_cmd("Page.captureScreenshot", {"format": "png", "fromSurface": True})
        image_data = base64.b64decode(screenshot['data'])
        with open(rf"{exhibit_image_folder}\Exhibit Output\{id_number}.png", "wb") as f:
            f.write(image_data)
    except Exception as e:
        print(f"\tError writing full page screenshot: {e}. Please check if the 'output path' Link is valid.")
        pass

    # HEADER
    header_exists = ((len(str(row_data.HEADER_LEFT)) > 0) and not pd.isna(row_data.HEADER_LEFT)) or ((len(str(row_data.HEADER_MIDDLE)) > 0) and not pd.isna(row_data.HEADER_MIDDLE)) or ((len(str(row_data.HEADER_RIGHT)) > 0) and not pd.isna(row_data.HEADER_RIGHT))
    header_text = """<div style="font-family: 'Times New Roman'; font-size:12px; display:flex; justify-content: space-between; align-items: center; width: 100%; box-sizing: border-box;margin-left:20px; margin-right:20px;">{{HEADER}}</div>"""
    if header_exists:
        header_text = header_text.replace("{{HEADER}}",'{{HEADER_LEFT}}{{HEADER_MIDDLE}}{{HEADER_RIGHT}}')
        if len(str(row_data.HEADER_LEFT)) > 0 and not pd.isna(row_data.HEADER_LEFT):
            header_text = header_text.replace("{{HEADER_LEFT}}","<span style=\'text-align:left;\'>" + str(f"{row_data.HEADER_LEFT}") + "</span>")
        else:
            header_text = header_text.replace("{{HEADER_LEFT}}","")
        if len(str(row_data.HEADER_MIDDLE)) > 0 and not pd.isna(row_data.HEADER_MIDDLE):
            header_text = header_text.replace("{{HEADER_MIDDLE}}","<span style=\'text-align:center;\'>" + str(f"{row_data.HEADER_MIDDLE}") + "</span>")
        else:
            header_text = header_text.replace("{{HEADER_MIDDLE}}","")
        if len(str(row_data.HEADER_RIGHT)) > 0 and not pd.isna(row_data.HEADER_RIGHT):
            header_text = header_text.replace("{{HEADER_RIGHT}}","<span style=\'text-align:right;\'>" + str(f"{row_data.HEADER_RIGHT}") + "</span>")
        else:
            header_text = header_text.replace("{{HEADER_RIGHT}}","")
    else:
        header_text = header_text.replace("{{HEADER}}","")

    # FOOTER
    footer_text = """<div style="width:100%; font-family: 'Times New Roman'; font-size:12px; text-align:center; padding-top:2px;">{{FOOTER}}"""+f"{row_data.LABEL} {label_number} | Page <span class='pageNumber'></span></div>"
    if len(str(row_data.FOOTER)) > 0 and not pd.isna(row_data.FOOTER):
        Footer_Text = str(row_data.FOOTER)
        Footer_Text = Footer_Text.replace('{','').replace('}','')
        footer_text = f"<div style=\"width:100%; font-family: 'Times New Roman'; font-size:12px; text-align:center; padding-top:2px;\">"+f"{Footer_Text} | {row_data.LABEL} {label_number} - Page <span class='pageNumber'></span></div>"
    else:
        footer_text = f"<div style=\"width:100%; font-family: 'Times New Roman'; font-size:12px; text-align:center; padding-top:2px;\">"+f"{row_data.LABEL} {label_number} - Page <span class='pageNumber'></span></div>"

    # PRINT TO PDF
    if str(f"{row_data.ORIENTATION}").upper().strip() == "LANDSCAPE":
        pdf = driver.execute_cdp_cmd(
            "Page.printToPDF",
            {"printBackground": True, "paperWidth": 11, "paperHeight": 8.5, "marginTop": 0.5, "marginBottom": 0.5, "marginLeft": 0.5, "marginRight": 0.5,
                "displayHeaderFooter": True, "headerTemplate": header_text, "footerTemplate": footer_text})
    else:
        pdf = driver.execute_cdp_cmd(
            "Page.printToPDF",
            {"printBackground": True, "paperWidth": 8.5, "paperHeight": 11, "marginTop": 0.5, "marginBottom": 0.5, "marginLeft": 0.5, "marginRight": 0.5,
                "displayHeaderFooter": True, "headerTemplate": header_text, "footerTemplate": footer_text, })

    # SAVE PDF
    try:
        with open(rf"{output_path}\{str(row_data.LABEL)} {str(label_number)}\{str(row_data.LABEL)} {str(label_number)}.pdf", "wb") as f:
            f.write(base64.b64decode(pdf['data']))
            Exhibits_PDFs.add(rf"{output_path}\{str(row_data.LABEL)} {str(label_number)}\{str(row_data.LABEL)} {str(label_number)}.pdf")
    except Exception as e:
        print(f"\tError writing PDF file: {e}. Please check if the 'output path' Link is valid.")
        pass
    try:
        with open(rf"{exhibit_image_folder}\Exhibit Output\{id_number}.pdf", "wb") as f:
            f.write(base64.b64decode(pdf['data']))
    except Exception as e:
        print(f"\tError writing exhibit output PDF file: {e}. Please check if the Exhibit Image Link is valid.")
        pass

    driver.quit()

    #delete temporary files
    try:
        os.remove(rf"{output_path}\{str(row_data.LABEL)} {str(label_number)}\{str(row_data.LABEL)} {str(label_number)}.html")
        os.remove(rf"{exhibit_image_folder}\Exhibit Output\{id_number}.html")
        inter_image_files = []
        for file in os.listdir(exhibit_image_folder + fr"\Exhibit Output"):
            if file.startswith(f"{id_number} Inter") and (file.endswith(".png") or file.endswith(".pdf")):
                inter_image_files.append(file)
        for image_file in inter_image_files:
            os.remove(exhibit_image_folder + fr"\Exhibit Output\{image_file}")

    except Exception as e:
        print(f"\tError deleting temporary files: {e}.")
        pass

# CREATE EXHIBITS
for row_data in Exhibit_Data.itertuples():
    create_exhibit(row_data)

# MERGE PDFs
merger = PdfMerger()
for pdf in Exhibits_PDFs:
    merger.append(pdf)
try:
    merger.write(fr"{output_path}\Exhibits.pdf")
except Exception as e:
    print(f"\tError writing merged PDF file: {e}. Please check if the 'path' Link is valid and that the file is not open elsewhere.")
    pass
merger.close()

Creating Exhibit 1
Creating Exhibit 2
Creating Exhibit 4
Creating Exhibit 5
Creating Exhibit 6
Creating Exhibit 7
Creating Exhibit 3
